### 1. 학습 목표
- Word2Vec의 두 가지 모델(CBOW, Skip-gram)의 작동 원리를 설명할 수 있다.
- 중심 단어와 주변 단어를 이용한 신경망 학습 과정을 이해한다.

### 2. 용어 정리
- **정의 (CBOW, Continuous Bag-of-Words):** 주변 단어(Context)들을 입력으로 받아 중심 단어(Target)를 예측하는 학습 방식.
- **사용 시나리오:** 전체적인 문맥이 주어졌을 때 빈칸에 들어갈 단어를 맞추는 문제와 유사.
- **구체적 예제:** [the, cat, (?), on, the] → (?) = "sits".

- **정의 (Skip-gram):** 중심 단어를 입력으로 받아 주변 단어들을 예측하는 학습 방식.
- **사용 시나리오:** 특정 단어가 주어졌을 때 그 주변에 어떤 단어들이 올지 추론할 때 활용.
- **구체적 예제:** [sits] → 주변 단어 {cat, on} 예측.

### 3. 이론적 배경
Word2Vec은 얕은 신경망(Input - Projection - Output) 구조를 가집니다. 학습이 끝나면 신경망의 가중치 행렬 $W$가 각 단어의 임베딩 벡터가 됩니다. 대규모 코퍼스를 통해 단어의 의미적 공간을 구축합니다.

### 4. 예제 코퍼스
- 문장: "I love cats"
- 윈도우 크기(Window Size): 1
- **Skip-gram 학습 데이터 쌍 (중심, 주변):** (love, I), (love, cats)

### 5. 수식 유도 및 직접 계산
**Softmax 기반 확률 계산:**
$$P(w_o | w_i) = \frac{\exp(\mathbf{v}_o^\top \mathbf{v}_i)}{\sum_{w=1}^V \exp(\mathbf{v}_w^\top \mathbf{v}_i)}$$

**계산 시나리오 (단어 $i$와 $o$의 유사도 점수가 $2$이고, 전체 점수 합이 $10$일 때):**
1. 분자: $\exp(2) \approx 7.39$
2. 분모(모든 단어의 지수 합): $10$ (가정)
3. 예측 확률: $P = 7.39 / 10 = 0.739$

기호 설명:
- $\mathbf{v}_i$: 입력 단어(중심 단어)의 벡터
- $\mathbf{v}_o$: 출력 단어(주변 단어)의 벡터
- $\mathbf{v}_w$: 전체 어휘 사전($V$) 내에 존재하는 임의의 단어 $w$의 벡터
- $V$: 전체 어휘 사전의 크기 (단어의 총 개수)

### 6. 비교 분석: CBOW vs Skip-gram
| 모델 | CBOW | Skip-gram |
| :--- | :--- | :--- |
| **예측 방향** | 주변 단어 → 중심 단어 | 중심 단어 → 주변 단어 |
| **학습 속도** | 빠름 | 상대적으로 느림 |
| **성능 특징** | 흔한 단어 예측에 강함 | 희귀 단어 및 대량 데이터에 강함 (일반적 권장) |

### 실습 Gensim 라이브러리

In [ ]:
%pip install gensim

In [ ]:
from gensim.models import Word2Vec

In [ ]:
corpus = [
    'i love nlp',
    'i love machine learning!',
    'nlp is fun^^',
    'machine learning is powerful',
    'i enjoy deep learning',
    'natural langauge processing is interesting'
]
import re
sentences = [re.findall(r'\w+',text.lower()) for text in corpus]
# word2vec 모델 학습
model = Word2Vec(
    sentences=sentences,
    vector_size=100,
    window=3,  # 주변단어 범위
    min_count=1 ,  #최소등장횟수
    sg = 1  # 0: CBOW  1=skip-gram
)
# 단어백터 확인
vector = model.wv['love']
# 유사단어 찾기
model.wv.most_similar('learning')


In [ ]:
# 유사도계산
model.wv.similarity('fun','learning')

In [ ]:
model.wv.index_to_key
model.save('customword2vec.model')
loaded_model = Word2Vec.load('customword2vec.model')

In [ ]:
# 한국어 지원
import pandas as pd
df = pd.read_csv('daum_movie_review.csv')
corpurs = df['review'][:100]
corpurs = corpurs.to_numpy()

In [35]:
import re
from konlpy.tag import Okt
okt = Okt()
# 전처리
sentences = [re.sub(r'[^가-힣\s]','',text) for text in corpurs]
# 형태소분리
def korean_token(text):
    return [word for word,_  in okt.pos(text,stem=True) if len(word)> 1]
    

sentences = [korean_token(doc) for doc in sentences]
# 나머지는 동일하게 적용
# word2vec 모델 학습
model = Word2Vec(
    sentences=sentences,
    vector_size=1000,
    window=3,  # 주변단어 범위
    min_count=5 ,  #최소등장횟수
    sg = 1  # 0: CBOW  1=skip-gram
)
# 유사단어 찾기
model.wv.most_similar('영화')

[('가다', 0.10001443326473236),
 ('있다', 0.0833105519413948),
 ('다음', 0.07395768165588379),
 ('같다', 0.06599206477403641),
 ('모르다', 0.05515427887439728),
 ('자다', 0.054868876934051514),
 ('나오다', 0.05061757192015648),
 ('그래도', 0.04825276881456375),
 ('재밌다', 0.044291913509368896),
 ('정말', 0.02504456602036953)]

In [23]:
sentences

['돈 들인건 티가 나지만 보는 내내 하품만',
 '몰입할수밖에 없다 어렵게 생각할 필요없다 내가 전투에 참여한듯 손에 땀이남',
 '이전 작품에 비해 더 화려하고 스케일도 커졌지만 전국 맛집의 음식들을 한데 모은 것까지는 좋았으나 이걸 모두 한 그릇에 섞어버린 듯한 느낌 그래도 다음 작품을 기대하게 만든다',
 '이 정도면 볼만하다고 할 수 있음',
 '재미있다',
 '나는 재밌게 봄',
 '점은 줄 수 없냐',
 '헐다 죽었어나중에 앤트맨 보다가도 깜놀',
 '충격 결말',
 '응집력',
 '개연성은 무시해라 액션을 즐겨라 스타로드가 이끌어준다 각각의 영웅들을 즐겨라 그리고 단적인 신념이 얼마나 부질없는지 보셔라',
 '내가졸라이상하네',
 '대박',
 '정말 지루할틈없이 넘잘만들었다 역시 대단하다',
 '역시 어벤져스',
 '마지막에 누구한테 연락한거지 궁금',
 '다음 편이 궁굼해지네요',
 '안잼있는사람 있음',
 '타노스 개갞기',
 '잘 만들었다 지루할 틈이 없네',
 '이제는 지겨워서 못보겠다',
 '롱턱 타노스의  장갑이 참 맘에 듬  아이언 맨과 토르 닥터만 생고생하고  가디언즈 오브 갤럭시 들 때문에  손해가 크다고 봄  들짐승 하고 칡뿌리 같은 캐릭이  재미를 더해줄줄 알았으나  아쉬움  분노의 상징 헐크가 겁을 먹다니로키의 초반 출연뿐이 서운하지만  본 영화에 이어질 내용에 적합하지 않은지 서두에  죽는부분으로 마무리 됨은 심섬한 충격임   다음편에 헬라가 다시 나올까',
 '내 인생 명작',
 '와진짜 개쪄는 인피니티워몇번을 봐도 개지린다너무잼있다어벤져스도 너무 기대된다빨리 년이왔으면 좋겠다',
 '종합선물세트지만 가장 중요한 마무리가 없는 예고편',
 '기대만큼 실망도크네요 임펙트도없고  내용도 그닥 너무억지',
 '이 영화를 보고나서 예전 영화 왓치맨이 생각나더군요 평화를 위해선 불특정 다수의 희생이 동반되어야 한다는 빌런들의 질서를 잡고자 하는 정의감이 영웅들의 여러 불특정 다수의 희생자를 만들고 이룬 질서와 뭐가 다를까란 의문을